In [ ]:
import uproot
import numpy as np
import matplotlib.pyplot as plt
import glob
import os
import re
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

# ============================================================
# CONFIGURATION PARAMETERS
# ============================================================
RUN_SEL = None  # None = tutti i file; intero = solo quel run ID

MAX_DEBUG_PLOTS = 0  # numero di waveform da plottare per debug

PLOT_ATTENUATION = True  # calcola e mostra log(Q_L/Q_R) e cariche

# ============================================================
# DIGITIZER, PMT & CABLE PARAMETERS
# ============================================================
SAMPLE_RATE_GS = 2.5
DT_NS = 1.0 / SAMPLE_RATE_GS    # = 0.4 ns/sample
RECORD_LENGTH = 1024
ADC_BITS = 12
V_PP_MV = 1000.0
BASELINE_ADC = 3800
NOISE_RMS_ADC = 3.5

TRIGGER_OFFSET_NS = 150.0

# PMT Properties — Hamamatsu H14603-103
G = 3.0e5
R = 50.0
e_charge = 1.602e-19
tr_pmt = 0.6
TTS_FWHM_NS = 0.3
TTS_SIGMA_NS = TTS_FWHM_NS / 2.355
SPE_RES = 0.40

# RG-58 Cable Properties
CABLE_LENGTH_M = 20.0
ATTENUATION_DB = (15.0 / 100.0) * CABLE_LENGTH_M
TRANSMISSION_FACTOR = 10.0 ** (-ATTENUATION_DB / 20.0)

tr_cable = 0.14 * CABLE_LENGTH_M
tr_total = np.sqrt(tr_pmt**2 + tr_cable**2)
sigma_total = tr_total / 2.22

amp_SPE_mV_ideal = - (e_charge * G * R) / (sigma_total * 1e-9 * np.sqrt(2 * np.pi)) * 1e3
amp_SPE_mV_attenuated = amp_SPE_mV_ideal * TRANSMISSION_FACTOR

QUICK_INT_START_NS = 250.0
QUICK_INT_END_NS   = 410.0

DEBUG_XLIM_START_NS = 250.0
DEBUG_XLIM_END_NS   = 410.0

def mv_to_adc(voltage_mv):
    lsb = V_PP_MV / (2**ADC_BITS)
    adc_float = BASELINE_ADC + (voltage_mv / lsb)
    adc_noisy = adc_float + np.random.normal(0, NOISE_RMS_ADC, len(adc_float))
    return np.clip(np.round(adc_noisy), 0, (2**ADC_BITS)-1).astype(np.uint16)

def get_quantum_efficiency(wavelengths_nm):
    """QE curve for Hamamatsu H14603-103 (bialkali photocathode)."""
    wl_points = np.array([200.0, 250.0, 300.0, 350.0, 380.0, 400.0, 420.0,
                          450.0, 500.0, 550.0, 600.0, 650.0, 700.0])
    qe_points = np.array([0.18,  0.25,  0.30,  0.34,  0.35,  0.34,  0.32,
                          0.28,  0.18,  0.08,  0.03,  0.005, 0.00])
    return np.interp(wavelengths_nm, wl_points, qe_points, left=0.0, right=0.0)

# ============================================================
# BATCH READING AND OUTPUT PREPARATION
# ============================================================
input_dir = "/Users/benussi/Testbeam2026_WC_single/data"
root_files = glob.glob(os.path.join(input_dir, "*.root"))

output_dir = "/Users/benussi/Testbeam2026_WC_single/data/"
os.makedirs(output_dir, exist_ok=True)

if len(root_files) == 0:
    print(f"No .root files found in {input_dir}. Please check the path.")
    exit()

print(f"Found {len(root_files)} ROOT files. Starting massive digitization...")
print(f" -> Sample rate: {SAMPLE_RATE_GS} GS/s  (dt = {DT_NS:.3f} ns/sample, record {RECORD_LENGTH} samples = {RECORD_LENGTH*DT_NS:.1f} ns)")
print(f" -> Trigger offset: {TRIGGER_OFFSET_NS} ns  (signal centroid expected around sample {int(TRIGGER_OFFSET_NS/DT_NS)})")
print(f" -> PMT: Hamamatsu H14603-103 (G={G:.1e}, TTS={TTS_FWHM_NS} ns FWHM, SPE res={SPE_RES*100:.0f}%)")
print(f" -> Simulated Cable: {CABLE_LENGTH_M}m RG-58 (-{ATTENUATION_DB:.1f} dB, Tr_cable={tr_cable:.1f} ns)")

time_axis = np.arange(0, RECORD_LENGTH * DT_NS, DT_NS)
mock_catalog = []
events_summary = []
current_run_id = 1000
debug_plots_saved = 0
quick_analysis_data = []

window_start = int(QUICK_INT_START_NS / DT_NS)
window_end   = int(QUICK_INT_END_NS / DT_NS)

# Modulo singolo: PMT_ID 0 = PMT_L (ch_A), PMT_ID 1 = PMT_R (ch_B)
N_PMT = 2
ch_A, ch_B = 0, 1

for file_path in sorted(root_files):

    if RUN_SEL is not None and current_run_id != RUN_SEL:
        current_run_id += 1
        continue

    basename = os.path.basename(file_path)

    # Parsing nome file: sim_Z<Z>_Y<Y>_X<X>_<DEG>Deg.root
    match = re.search(r'Z([+-]?\d+\.?\d*)_Y([+-]?\d+\.?\d*)_X([+-]?\d+\.?\d*)_([+-]?\d+)Deg', basename, re.IGNORECASE)
    if match:
        z_val   = float(match.group(1))
        y_val   = float(match.group(2))
        x_val   = float(match.group(3))
        deg_val = float(match.group(4))
        y_cm = y_val / 10.0
        x_cm = x_val / 10.0
        mod_name = f"Single Module | Y={y_val:.1f} mm X={x_val:.1f} mm Theta={deg_val:.0f}deg"
    else:
        print(f"Unrecognized file name format for {basename}. Skipping.")
        continue

    print(f" -> Processing Run {current_run_id} | {mod_name}")

    events_data_per_file = {}

    try:
        file = uproot.open(file_path)
        tree = file["Fotoni"]
        df = tree.arrays(["EventID", "Arrival_Time_ns", "PMT_ID", "E_Hit_eV"], library="pd")
    except Exception as e:
        print(f"    [!] ROOT tree reading error: {e}")
        events_summary.append({'File_Root': basename, 'Valid_Events_Min_1_Photon': "Read Error"})
        current_run_id += 1
        continue

    if df.empty:
        print("    [!] No photons recorded in this run.")
        events_summary.append({'File_Root': basename, 'Valid_Events_Min_1_Photon': 0})
        current_run_id += 1
        continue

    event_ids = np.sort(df.EventID.unique())
    num_events_with_photons = len(event_ids)
    events_summary.append({'File_Root': basename, 'Valid_Events_Min_1_Photon': num_events_with_photons})

    qa_list_run = []
    qb_list_run = []

    for ev_id in event_ids:
        event_dict = {}
        photon_counts_debug = {}
        hits_ev = df[df.EventID == ev_id]

        # Digitalizza i 2 PMT (ch_0 = PMT_L, ch_1 = PMT_R)
        for pmt_id in range(N_PMT):
            photons_pmt = hits_ev[hits_ev.PMT_ID == pmt_id]
            photon_counts_debug[pmt_id] = len(photons_pmt)

            if photons_pmt.empty:
                event_dict[f'ch_{pmt_id}'] = mv_to_adc(np.zeros_like(time_axis))
                continue

            arrival_times = photons_pmt.Arrival_Time_ns.values
            energies_ev   = photons_pmt.E_Hit_eV.values

            wavelengths_nm = 1240.0 / energies_ev
            qe_probs = get_quantum_efficiency(wavelengths_nm)
            accepted_mask = np.random.rand(len(qe_probs)) < qe_probs
            accepted_arrival_times = arrival_times[accepted_mask]

            voltage = np.zeros_like(time_axis)
            for t_hit in accepted_arrival_times:
                tts_jitter = np.random.normal(0.0, TTS_SIGMA_NS)
                t_shifted = t_hit + TRIGGER_OFFSET_NS + tts_jitter
                spe_gain_factor = np.random.normal(1.0, SPE_RES)
                if spe_gain_factor <= 0:
                    continue
                voltage += amp_SPE_mV_attenuated * spe_gain_factor * np.exp(-0.5 * ((time_axis - t_shifted) / sigma_total)**2)

            event_dict[f'ch_{pmt_id}'] = mv_to_adc(voltage)

        if PLOT_ATTENUATION:
            lsb = V_PP_MV / (2**ADC_BITS)
            sigA_mV = (event_dict[f'ch_{ch_A}'].astype(float) - BASELINE_ADC) * lsb
            sigB_mV = (event_dict[f'ch_{ch_B}'].astype(float) - BASELINE_ADC) * lsb
            qA_val = -np.sum(sigA_mV[window_start:window_end])
            qB_val = -np.sum(sigB_mV[window_start:window_end])
            if qA_val > 0 and qB_val > 0:
                qa_list_run.append(qA_val)
                qb_list_run.append(qB_val)

        # DEBUG PLOT
        if debug_plots_saved < MAX_DEBUG_PLOTS:
            fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True, sharey=True)
            fig.suptitle(f"DEBUG: Run {current_run_id} - Event {ev_id} ({basename})", fontsize=13)
            lsb = V_PP_MV / (2**ADC_BITS)
            pmt_labels = ['PMT_L (ch_0)', 'PMT_R (ch_1)']
            for pmt_id in range(N_PMT):
                ax = axes[pmt_id]
                signal_mv = (event_dict[f'ch_{pmt_id}'].astype(float) - BASELINE_ADC) * lsb
                ax.plot(time_axis, signal_mv, color='darkcyan', linewidth=1.5)
                ax.axhline(0.0, color='red', linestyle='--', alpha=0.7)
                ax.axvline(TRIGGER_OFFSET_NS, color='magenta', linestyle=':', alpha=0.7)
                ax.set_title(f"{pmt_labels[pmt_id]} ({photon_counts_debug[pmt_id]} ph)", fontsize=11)
                ax.set_xlim(DEBUG_XLIM_START_NS, DEBUG_XLIM_END_NS)
                ax.set_xlabel("Time (ns)")
                ax.grid(True, linestyle=':', alpha=0.6)
            axes[0].set_ylabel("Amplitude (mV)")
            plt.tight_layout()
            plt.show()
            debug_plots_saved += 1

        events_data_per_file[f'run_{current_run_id}_ev_{ev_id}'] = event_dict

    if PLOT_ATTENUATION and len(qa_list_run) > 0:
        qa_arr = np.array(qa_list_run)
        qb_arr = np.array(qb_list_run)
        log_ratios_raw = np.log(qa_arr / qb_arr)
        quick_analysis_data.append({
            'Module': mod_name,
            'Y_cm': y_cm,
            'X_cm': x_cm,
            'Theta_deg': deg_val,
            'Log_Ratio': np.mean(log_ratios_raw),
            'Err_Log_Ratio': np.std(log_ratios_raw) / np.sqrt(len(log_ratios_raw)),
            'Q_L': np.mean(qa_arr),
            'Err_Q_L': np.std(qa_arr) / np.sqrt(len(qa_arr)),
            'Q_R': np.mean(qb_arr),
            'Err_Q_R': np.std(qb_arr) / np.sqrt(len(qb_arr)),
            'Num_Events': len(qa_list_run)
        })

    npz_filename = f"{os.path.splitext(basename)[0]}.npz"
    npz_path = os.path.join(output_dir, npz_filename)
    np.savez_compressed(npz_path, **events_data_per_file)

    mock_catalog.append({
        'run': current_run_id,
        'descrizione': mod_name,
        'status': 'ok',
        'file_npz': npz_filename,
        'valid_events': num_events_with_photons
    })
    current_run_id += 1

# ============================================================
# QUICK ATTENUATION ANALYSIS
# ============================================================
if PLOT_ATTENUATION and len(quick_analysis_data) > 0:
    df_qa = pd.DataFrame(quick_analysis_data).sort_values(by='X_cm')

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle("Quick Analysis: Charge & Attenuation vs Beam X Position", fontsize=14)

    for theta, grp in df_qa.groupby('Theta_deg'):
        grp = grp.sort_values('X_cm')
        X_arr = grp['X_cm'].values

        axes[0].errorbar(X_arr, grp['Log_Ratio'], yerr=grp['Err_Log_Ratio'],
                         fmt='o-', label=f'{theta:.0f}°')
        axes[1].errorbar(X_arr, grp['Q_L'], yerr=grp['Err_Q_L'],
                         fmt='s-', label=f'{theta:.0f}°')
        axes[2].errorbar(X_arr, grp['Q_R'], yerr=grp['Err_Q_R'],
                         fmt='^-', label=f'{theta:.0f}°')

    axes[0].set_title(r"$\ln(Q_L / Q_R)$")
    axes[1].set_title(r"$Q_L$ (PMT_L)")
    axes[2].set_title(r"$Q_R$ (PMT_R)")
    for ax in axes:
        ax.set_xlabel("Beam X Position (cm)")
        ax.grid(True, linestyle=':', alpha=0.7)
        ax.legend(fontsize=9, title='Theta')
    plt.tight_layout()
    plt.show()

# Save catalogs
if RUN_SEL is None:
    csv_path = os.path.join(output_dir, "mc_runs_catalog.csv")
    pd.DataFrame(mock_catalog).to_csv(csv_path, index=False)
    print(f"\n[OK] Run catalog saved to: {csv_path}")

    summary_path = os.path.join(output_dir, "eventi_con_fotoni_summary.csv")
    pd.DataFrame(events_summary).to_csv(summary_path, index=False)
    print(f"[OK] Valid events summary saved to: {summary_path}")